# SmolVLA Evaluation — Corrected

**Fix**: lerobot-eval supports LoRA natively via .
Previous runs failed because this flag was missing.


In [ ]:
# Cell 0: Install
!pip install -q "lerobot[smolvla,peft,libero]" peft

import os, torch
os.environ["MUJOCO_GL"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cell 1: Quick sanity check — verify adapter_config.json has base_model_name_or_path
from huggingface_hub import hf_hub_download
import json

LORA_REPO = "dennywu2966/smolvla-libero-object-lora"

try:
    cfg_path = hf_hub_download(LORA_REPO, "adapter_config.json")
    with open(cfg_path) as f:
        adapter_cfg = json.load(f)
    base = adapter_cfg.get("base_model_name_or_path", "MISSING")
    print(f"Adapter config OK. Base model: {base}")
except Exception as e:
    print(f"adapter_config.json load failed: {e}")
    print("Checkpoint may not contain LoRA adapter files — check HF Hub repo contents")


In [ ]:
# Cell 2: Evaluate fine-tuned SmolVLA LoRA
# Key fix: --policy.use_peft=true tells lerobot to load LoRA adapter from pretrained_path
import subprocess, time, os

LORA_REPO = "dennywu2966/smolvla-libero-object-lora"
eval_env = {**os.environ, "MUJOCO_GL": "egl", "TOKENIZERS_PARALLELISM": "false"}

start = time.time()
proc = subprocess.Popen([
    "lerobot-eval",
    f"--policy.path={LORA_REPO}",
    "--policy.use_peft=true",          # <-- THE FIX
    "--policy.device=cuda",
    "--env.type=libero",
    "--env.task=libero_object",
    "--eval.n_episodes=20",
    "--eval.batch_size=1",
], env=eval_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

output_lines = []
for line in proc.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

proc.wait(timeout=10800)
elapsed = time.time() - start
print(f"\nExit code: {proc.returncode} | Time: {elapsed/60:.1f} min")


In [ ]:
# Cell 3: Evaluate official HuggingFaceVLA SmolVLA (sanity check / upper bound)
# This is the fully fine-tuned reference model from HuggingFaceVLA
import subprocess, time, os

OFFICIAL_REPO = "HuggingFaceVLA/smolvla_libero"
eval_env = {**os.environ, "MUJOCO_GL": "egl", "TOKENIZERS_PARALLELISM": "false"}

start = time.time()
proc = subprocess.Popen([
    "lerobot-eval",
    f"--policy.path={OFFICIAL_REPO}",
    "--policy.device=cuda",
    "--env.type=libero",
    "--env.task=libero_object",
    "--eval.n_episodes=20",
    "--eval.batch_size=1",
], env=eval_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

output_lines_official = []
for line in proc.stdout:
    print(line, end="", flush=True)
    output_lines_official.append(line)

proc.wait(timeout=10800)
elapsed = time.time() - start
print(f"\nExit code: {proc.returncode} | Time: {elapsed/60:.1f} min")


In [ ]:
# Cell 4: Parse results and build comparison table
import glob, json, sys
sys.path.insert(0, "/kaggle/working/vlm-vla/src")
from vlm_vla.eval_engine import EvalReport, TaskResult, compare_reports

# lerobot-eval prints aggregated metrics to stdout and saves JSON to outputs/
result_files = sorted(glob.glob("outputs/*/eval_info.json") + glob.glob("outputs/eval_*.json"))
print(f"Result files: {result_files}")

results = {}
for rf in result_files:
    with open(rf) as f:
        data = json.load(f)
    print(f"\n--- {rf} ---")
    print(json.dumps(data.get("overall", data), indent=2)[:2000])
    results[rf] = data

if not results:
    print("No result files — check stdout output in Cell 2 for success rate lines")
    # Parse from stdout as fallback
    import re
    for line in output_lines:
        if "success" in line.lower() or "%" in line:
            print(line.rstrip())


In [ ]:
# Cell 5: Final comparison table
# Fill in actual per-task success rates from Cell 4 output
# Template — adapt task names and rates from actual output

import sys
sys.path.insert(0, "/kaggle/working/vlm-vla/src")
from vlm_vla.eval_engine import EvalReport, TaskResult, compare_reports

# SmolVLA zero-shot reference (confirmed: 0% all tasks)
smolvla_zs = EvalReport(
    model_name="SmolVLA-ZeroShot",
    task_suite="libero_object",
    results=[TaskResult(f"task_{i}", 0.0, 20, 400.0, {"timeout": 20}) for i in range(10)],
)

# OpenVLA 7B published reference
openvla_ref = EvalReport(
    model_name="OpenVLA-7B-FT",
    task_suite="libero_object",
    results=[TaskResult(f"task_{i}", 0.884, 20, 120.0, {}) for i in range(10)],
)

# TODO: fill from Cell 4 output
# smolvla_ft_results = [TaskResult("libero_object_task_0", 0.XX, 20, ?, {}), ...]
# smolvla_ft = EvalReport("SmolVLA-LoRA-20k", "libero_object", smolvla_ft_results)
# print(compare_reports(smolvla_ft, smolvla_zs, openvla_ref))

print("Reference: SmolVLA zero-shot baseline")
print(smolvla_zs.to_table())
print("\nReference: OpenVLA-7B published")
print(openvla_ref.to_table())
